### Multi- Head Attention
trainable weights = wk,wq,wv 
are increase 
head = 2 
wk , wq , wv -> each 2 matrix generate 
concate the final context vector 

single head attention-> context vector dim=2
2 head attention ->> context vector dim=2 (2 generate)n--> concate = final context vector = dim =4


In [20]:
import torch
inputs=torch.tensor([[0.22,0.44,0.56], # your 
                    [0.33,-0.144,0.756], # journey
                    [0.22,0.344,0.856], # starts
                    [-0.722,-0.644,-0.356], # with
                    [0.022,0.744,0.1356], # one
                    [0.122,0.044,0.956]]) # step 

In [21]:
batch=torch.stack((inputs,inputs),dim=0)
batch.shape ## 2 batch of embedding vectors 

torch.Size([2, 6, 3])

In [22]:
d_in=3
d_out=2
contex_length=batch.shape[1]
dropout=0.5
num_head=2
contex_length

6

In [23]:
import torch
import torch.nn as nn

In [24]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

In [25]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [26]:
instance_multi_head=MultiHeadAttentionWrapper(d_in,d_out,contex_length,dropout,num_head)
context_vec=instance_multi_head(batch)
context_vec

tensor([[[-0.2457,  0.2065,  1.0711, -0.1512],
         [-0.2157,  0.3084,  0.8601, -0.4424],
         [-0.0626,  0.1360,  0.0000,  0.0000],
         [ 0.1080,  0.0493,  0.4589, -0.2226],
         [-0.0362,  0.0787,  0.1555,  0.0604],
         [ 0.0426,  0.0246, -0.1857, -0.1006]],

        [[-0.2457,  0.2065,  0.0000,  0.0000],
         [-0.0954,  0.2074,  0.3563, -0.3713],
         [-0.0741,  0.1299,  0.2318, -0.2415],
         [ 0.0049,  0.2499,  0.7798, -0.3343],
         [-0.1693,  0.1833,  0.2844, -0.0739],
         [-0.0988,  0.0928, -0.0761, -0.0521]]], grad_fn=<CatBackward0>)